This script plots various aspects of the TrappedResonance objective function

In [4]:
import desc.io
from desc.objectives import (
    TrappedResonance
)
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

In [3]:
# User Inputs #
eq = desc.io.load("equil_G1600_DESC_fixed.h5")
rhos = (np.linspace(0.1,0.9,3))**(1/2) # rho = sqrt(s)
alphas = np.linspace(0,2*np.pi,5)
KE_frac = np.array([1]) #did 0.001 before
pitch_invs = jnp.array([6.05])
N=0 # QA

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'equil_G1600_DESC_fixed.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
# Run objective function
eq_periodicity = (np.inf,np.inf,np.inf) # periodicity in zeta for these equilibrium to make rtz grid
grid = eq._get_rtz_grid( # returns rho, theta, zeta coordinate grid
    rhos, # radial
    np.array([0]), # poloidal (alpha in this case)
    np.array([0]), # toroidal (zeta in this case)
    coordinates="raz", # rho, alpha, zeta input coordinates
    period=eq_periodicity, # periodicity of coordinate (rho,alpha,zeta)
)
Psi = eq.compute("Psi",grid=grid)

obj = TrappedResonance(eq,rho=rhos,pitch_invs=pitch_invs,KE_frac=KE_frac,alpha=alphas,N=N,Psi=Psi['Psi'])
# obj = TrappedResonance(eq,rho=rhos,pitch_invs=pitch_invs,KE_frac=KE_frac,alpha=alphas,N=N)
obj.build()
out = obj.compute(eq.params_dict) # := (rho,lambda,well)

In [12]:
# Poincare plotting saving
poinc_plot = np.sum(out['poinc_plot'],axis=2)[:,0] # := (rho), summed over wells cuz why not
omega_arr = out['omega_arr'][:,0,0]

# Save objective function values to the firm3d directory for plotting with Poincare plots
np.savetxt('/Users/paullab/codes/firm3d_fork_10132025/firm3d/examples/trapped_map/obj_DESC.txt',poinc_plot)
np.savetxt('/Users/paullab/codes/firm3d_fork_10132025/firm3d/examples/trapped_map/freq_DESC.txt',omega_arr)
# np.savetxt('/Users/paullab/codes/firm3d_fork_10132025/firm3d/examples/trapped_map/tau_DESC.txt',out['tau_arr'][:,0,0,0])

In [ ]:
print(out['pitch_inv'])

In [ ]:
# Plot s vs. obj_out for each pitch inverse
for i in range(0,obj_out.shape[1]): # loop through each pitch angle
    plt.figure(i+1)
    plt.plot(rhos**2,obj_out[:,i,0]) # assuming we are only doing 1 energy for now
    plt.xlabel(r'$s$')
    plt.ylabel('Objective Function Value')
    plt.title('TrappedResonance Objective Function Value for Pitch Inverse = '+str(pitch_invs[i]))
    # plt.ylabel(r'$\omega_{\zeta}$')
    # plt.title(r'$\omega_{\zeta}$ Value for Pitch Inverse = '+str(pitch_invs[i]))
    plt.xlim([0.0,1.0])

In [ ]:
# Plot the value of the TrappedResonance objective (summing over the rho axis) vs. pitch
obj_out = out['obj']
num_valid_pitch = jnp.sum(obj_out!=0,axis=0) # for each surface, how many pitches were valid

obj_out_rhosum = np.sum(obj_out,axis=0) / num_valid_pitch
np.savetxt('obj_pitch_plot.txt',obj_out_rhosum[:,0])
plt.figure()
plt.plot(pitch_invs,obj_out_rhosum[:,0]/np.nanmax(obj_out_rhosum[:,0]))
plt.title("TrappedResonance Objective Function Averaged Over All Surfaces")
plt.xlabel("Pitch Inverse [T]")
plt.ylabel("Objective Function Value")

In [51]:
np.savetxt('obj_s_out.txt',out['obj'][:,0,0]) # value at all surfaces

In [ ]:
obj_val = out['obj'][:,0,0]
Y = rhos**2
X = np.linspace(0,np.pi,5)
Z = np.transpose(np.tile(obj_val, (5, 1)))
plt.contourf(X,Y,Z)
plt.colorbar(label='Objective Function Value')

MULTI-EQUILIBRIA PLOTTING

In [6]:
import desc.io
from desc.objectives import (
    TrappedResonance
)
from desc.equilibrium.coords import get_rtz_grid
import numpy as np
import jax.numpy as jnp


## Inputs
eq_dir = "Equilibria/"
equilibrias_desc = ["desc_eq_beta2.5_QA.h5",
            #    "equil_G1600_DESC_fixed_QA.h5",
            #    "equil_Helios_E0090-13_DESC_fixed_QA.h5",
            #    "equil_Helios_E0092_DESC_fixed_QA.h5",
            #    "desc_eq_new_QH_aScaling_QH.h5",
               ]
rhos = np.sqrt(np.linspace(0.1,0.9,20)) # rho = sqrt(s)
alphas = np.linspace(0,2*np.pi,2)
num_Bc = 3 # number of Bcrit to include
KE_frac = np.array([1])
##

file_dict = {}

for equilibria in equilibrias_desc:
    eq = desc.io.load(eq_dir+equilibria)
    config_id = equilibria[-4]
    if config_id == "H":
        N=1 # QH
    elif config_id == "A":
        N=0
    elif config_id == "I":
        print("not yet implemented")

    # Determine Bcrits
    rho_bc = np.linspace(0.1,0.9,9)
    alpha_bc = np.array([0])
    zeta_bc = np.linspace(0, 12 * np.pi, 50)
    grid = get_rtz_grid(eq, rho_bc, alpha_bc, zeta_bc, coordinates="raz")
    data = eq.compute(["min_tz |B|", "max_tz |B|"], grid=grid) # min,max |B| on each rho
    pitch_invs = jnp.linspace(
        np.mean(data['min_tz |B|']),
        np.mean(data['max_tz |B|']),
        num_Bc
        )

    # Run objective function and save output
    grid = eq._get_rtz_grid( # returns rho, theta, zeta coordinate grid
        rhos, # radial
        np.array([0]), # poloidal (alpha in this case)
        np.array([0]), # toroidal (zeta in this case)
        coordinates="raz", # rho, alpha, zeta input coordinates
    )
    Psi = eq.compute("Psi",grid=grid)
    obj = TrappedResonance(eq,rho=rhos,pitch_invs=pitch_invs,KE_frac=KE_frac,alpha=alphas,N=N,Psi=Psi['Psi'])
    # grid is set in init of TrappedResonance class
    obj.build()
    value = obj.compute(eq.params_dict)
    file_dict[equilibria] = {
        'name': equilibria,
        'Bc': pitch_invs,
        'f_tr2_desc': value,
        'N': N
    }
print(file_dict)
np.save('multiequil_plot/desc_out_multiequil',file_dict)

Precomputing transforms
{'desc_eq_beta2.5_QA.h5': {'name': 'desc_eq_beta2.5_QA.h5', 'Bc': Array([5.51005055, 5.79604329, 6.08203603], dtype=float64), 'f_tr2_desc': Array(-4.02089094e-12, dtype=float64), 'N': 0}}


In [ ]:
# Load in FIRM3D values
firm3d_out = np.load('multiequil_plot/firm3d_out.npy')
firm3d_names = []
firm3d_vals = []
config_id_plot = []
for key in firm3d_out:
    firm3d_names.append(key)
    firm3d_vals.append(firm3d_out[key]['val'])
    config_id_plot.append(key[-4])

# "Load" in DESC values
desc_out = np.load('multiequil_plot/desc_out_multiequil')
desc_vals = []
for key in desc_out:
    desc_vals.append(desc_out[key]['f_tr2_desc'])

In [ ]:
# Plotting
fig,ax = plt.subplots()
ax.set_xlabel(r'\Delta \rho')
ax.set_ylabel(r'Objective Function Value')
loop=0
for eq in firm3d_names:
    if config_id_plot == "H":
        ax.plot(firm3d_vals[loop],desc_vals[loop],'o')
    elif config_id_plot == "A":
        ax.plot(firm3d_vals[loop],desc_vals[loop],'*')
    elif config_id_plot == "I":
        ax.plot(firm3d_vals[loop],desc_vals[loop],'x')
    print("not yet implemented")
    loop+=1